In [1]:
# STEP 10.1 — Load trained Model 1
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_PATH = "outputs/model1_qlora/final"

compute_dtype = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
)

model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH,
)

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_PATH
)

model.eval()

print("Model 1 loaded successfully.")
print("Tokenizer loaded successfully.")


c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0904 22:36:22.329000 8420 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:18<00:00, 15.65it/s]


Model 1 loaded successfully.
Tokenizer loaded successfully.


In [4]:
# STEP 10.2 — Create inference function

def ask_model(messages):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move inputs to the model's device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Get only the newly generated tokens
    input_length = inputs["input_ids"].shape[-1]
    new_tokens = output[0][input_length:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


In [7]:
# STEP 10.6 — Quick behavior test

tests = [
    "I've had stomach pain since yesterday.",
    "I've been coughing for two weeks.",
    "I've been feeling dizzy lately."
]

for patient_message in tests:
    messages = [
        {
            "role": "user",
            "content": patient_message
        }
    ]

    response = ask_model(messages)

    print("=" * 60)
    print("Patient:", patient_message)
    print("Assistant:", response)

Patient: I've had stomach pain since yesterday.
Assistant: How would you describe the pain? Is it sharp, dull, or crampy? Is it constant or does it come and go? Is it worse after eating? Is it worse after eating certain foods? Is it worse after eating spicy foods? Is it worse after eating fatty foods? Is it worse after eating acidic foods? Is it worse after eating sweet foods? Is it worse after eating sour
Patient: I've been coughing for two weeks.
Assistant: Do you have any other symptoms? Any fever, chills, or night sweats? Any shortness of breath or chest pain? Any nausea or vomiting? Any diarrhea or constipation? Any headache or fatigue? Any joint pain or muscle pain? Any loss of appetite? Any weight loss? Any loss of sleep? Any loss of interest in activities? Any loss of interest in sex? Any loss
Patient: I've been feeling dizzy lately.
Assistant: How long have you been feeling this way? Is it constant? Do you have any other symptoms? Any nausea, vomiting, or headaches? Any other 